# Silver: parse the PDFs with `ai_parse_document`

Task `silver_parse` of the job `ingest_financebench`. Reads the parts in `bronze_pdf_parts` as a stream, runs the AI
parser on each one and saves the result in `silver_parsed`, **one row per part**:

| Column | Holds |
| --- | --- |
| `doc_name` | The document name, from the file name: `3M_2018_10K` for `.../pdfs/3M_2018_10K.pdf`. It is the `doc_name` FinanceBench's questions use |
| `path` | The PDF in the volume |
| `part_no`, `page_offset` | Which part of the PDF (bronze cuts long PDFs into parts of 50 pages), and how many pages come before it |
| `parsed` | The parser's output (VARIANT): the `elements` in reading order, each with a `type` (title, section_header, text, table as HTML, ...), its `content` and its page in `bbox[0].page_id`. The real page in the PDF is `page_offset + page_id + 1` |
| `parsed_at` | When it was parsed |

**Only complete parses are saved**: a part with elements and no `error_status`. The parser sometimes answers pages
with an error (`Request rate limit exceeded`, `Internal error`), and a part with missing pages would silently lose text.

**A stream** (`trigger(availableNow=True)`): a run parses the parts it has not seen, `pdfs_per_batch` PDFs at a time
(bronze writes one file per PDF), and stops. The checkpoint remembers what was read, so a rerun only parses new parts
and a saved part is never paid for twice. A part that failed is not tried again by a normal run: run with
`retry_failed` = `true` to read every part again (the saved ones are skipped). The last cell lists what is missing.

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("pdfs_per_batch", "2")  # PDFs parsed at once, small so the parser's rate limit is not hit
dbutils.widgets.dropdown("retry_failed", "false", ["false", "true"])  # true: read every part again (saved ones are skipped)

catalog = dbutils.widgets.get("catalog")
silver = f"{catalog}.{dbutils.widgets.get('silver_schema')}"
parts_table = f"{catalog}.{dbutils.widgets.get('bronze_schema')}.bronze_pdf_parts"
parsed_table = f"{silver}.silver_parsed"
checkpoint = f"/Volumes/{catalog}/{dbutils.widgets.get('silver_schema')}/checkpoints/parse"
pdfs_per_batch = int(dbutils.widgets.get("pdfs_per_batch"))

## Step 1: the catalog and the schema

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver}")

## Step 2: the table

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {parsed_table} (
  doc_name STRING, path STRING, part_no INT, page_offset INT, parsed VARIANT, parsed_at TIMESTAMP
)""")

## Step 3: the volume for the stream's checkpoint

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {silver}.checkpoints")

## Step 4: read the parts as a stream and parse them

The parse is billed and runs once per part: the filter and the saved column both read the same parser call.

In [ ]:
if dbutils.widgets.get("retry_failed") == "true":
    dbutils.fs.rm(checkpoint, True)  # the stream will read every part again

(
    spark.readStream.option("maxFilesPerTrigger", pdfs_per_batch).table(parts_table)
    .join(spark.table(parsed_table).select("path", "part_no"), ["path", "part_no"], "left_anti")  # skip the saved parts
    .select(
        F.regexp_extract("path", r"([^/]+)\.pdf$", 1).alias("doc_name"),  # .../pdfs/3M_2018_10K.pdf -> 3M_2018_10K
        "path", "part_no", "page_offset",
        F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
    )
    .filter(
        "coalesce(size(cast(parsed:document:elements AS array<variant>)), 0) > 0"  # it has elements
        " AND coalesce(size(cast(parsed:error_status AS array<variant>)), 0) = 0"  # and no page failed
    )
    .withColumn("parsed_at", F.current_timestamp())
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(parsed_table)
    .awaitTermination()
)

## What is saved, and what is missing

A missing part was not saved (its parse failed): run with `retry_failed` = `true` to parse it again.

In [ ]:
display(spark.sql(f"""
SELECT (SELECT count(*) FROM {parts_table}) AS parts, (SELECT count(*) FROM {parsed_table}) AS parts_parsed,
       (SELECT count(DISTINCT doc_name) FROM {parsed_table}) AS documents
"""))
display(spark.sql(f"""
SELECT regexp_extract(p.path, '([^/]+)\\\\.pdf$', 1) AS doc_name, p.part_no, p.pages
FROM {parts_table} p LEFT ANTI JOIN {parsed_table} s ON s.path = p.path AND s.part_no = p.part_no
ORDER BY doc_name, p.part_no
"""))